Bibliotecas necessárias.

In [13]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U datasets huggingface_hub openai

# Importar bibliotecas necessárias
import pandas as pd
import json
from google import genai
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from openai import OpenAI

Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [2]:
# Resgata o segredo cadastrado com o nome para o hugging face
def logon_hugging_face(hugging_api_key):
  try:
    # Recuperar valor da chave criada no huggin face e cadastrada no Secrets do Google Colab.
    hf_token = userdata.get(hugging_api_key)
    # Realiza o login no Hugging Face
    login(token=hf_token)
  except Exception as e:
    print(f"Erro ao recuperar token: {e}")

# Povoar dataframe de questões do huggin face.
def load_questions(dataset_id):
  # Carrega o dataset (geralmente ele vem dividido em 'train', 'test', etc.)
  ds = load_dataset(dataset_id)

  # Converte uma partição específica 'train' para um dataframe.
  df = pd.DataFrame(ds['train'])

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df.insert(0, 'num', range(1, len(df)+1))

  # retorno da Dataframe.
  return df

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_close_questions(df_close_questions, question_min, question_max):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df = df_close_questions.iloc[question_min:question_max].copy()
  return df

# Função para submeter uma questão à IA.
def question_submit(client_ai, model_id, question, question_type, alternatives):
    prompt_usuario = f"""
    Pessoa:
    Atue como um specialista Jurídico. Analise a questão da OAB abaixo:

    tipo_de_questao: {question_type}
    Questão: {question}
    Alternativas: {alternatives}

    Tarefas:
    1. Responda a questão escolhendo a letra correta (A, B, C ou D).
    2. Classifique a área de direito, com base na informação tipo_de_questao que pode está em inglês,
       e se for preciso traduza-a, para classificar a área.

    Responda no formato JSON a seguir:
    {{
      "resposta": "LETRA",
      "area": "NOME_DA_AREA"
    }}
    """

    chat = client_ai.chat.completions.create(
        messages=[{"role": "user", "content": prompt_usuario}],
        model=model_id,
        response_format={"type": "json_object"}
    )
    return json.loads(chat.choices[0].message.content)

# Comparar resposta com gabarito.
def compare_answers(response, answer_key):
  if response == answer_key:
    return 'Certo'
  else:
    return 'Errado'

Dataset do hugging face

In [6]:
# Chave salva no Secrets do Google Colab.
hugging_api_key = 'hugging_colab'
logon_hugging_face(hugging_api_key)

# Meu sub-grupo de questões e respostas.
# Subconjunto das questões: 319 a 424.
question_min = 319
question_max = 424

# Povoar dataframe de questões do huggin face.
dataset_id = 'eduagarcia/oab_exams'
df_close_questions = load_questions(dataset_id)

# Criar um dataset com meu subconjunto de questões.
df_my_close_questions = load_my_close_questions(df_close_questions, question_min, question_max)

Usar o gemini 3.1 flash para responder.

In [21]:
def client_ai_instance(name_api_key):
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # O uso do Secrets é uma alternativa para que chave de API não fique exposta no código.
  # Tal chave previamente criada no Google AI Studio ou no Console do Groq.
  # Observação: o nome da chave definido precisa ser o mesmo inclusive com diferenciação de letra maiúscula e minúscula.
  api_key = userdata.get(name_api_key)
  client_ai = None

  # Verificar se foi o groq ou google.
  if name_api_key == 'groq_api_key':
    client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=api_key
  )
  elif name_api_key == 'oprouter_api_key':
    client_ai = OpenAI(
      base_url="https://openrouter.ai/api/v1/",
      api_key=api_key
    )
  elif name_api_key == 'google_api_key':
    client_ai = genai.Client(api_key=api_key)
  else:
    print('chave não encontrada')
  return client_ai # Added the missing return statement

# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview de 2026.
model_id = 'llama-3.1-8b-instant'
client_ai = client_ai_instance('groq_api_key')

for index, row in df_my_close_questions.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    question = row['question']
    question_type = row['question_type']
    alternatives = " | ".join(row['choices']['text'])
    answer_key = row['answerKey']
    response = question_submit(client_ai, model_id, question, question_type, alternatives)
    response_ai = response.get("resposta")
    # Comparar com o gabarito
    status = compare_answers(response_ai, answer_key)
    df_my_close_questions['response_ai'] = response_ai
    df_my_close_questions['status'] = status

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kn7rg2zeee3t7f7drnrps2kb` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5319, Requested 719. Please try again in 380ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}